# Practical Collection Workflows

This notebook covers the Day 4 morning block:

**09:00-12:00 — Practical Collection Workflows**

- scheduling scrapers and collectors;
- monitoring breakage;
- logging and versioning;
- making collectors reproducible.

The point is to move from "a script that works once" to "a workflow that can be rerun, audited, monitored, and explained."

## 1. Where the Demo Run Files Went

The demo files from the test run were written to `/tmp`, not into the course repository.

For example, the test created a folder like:

```text
/tmp/methodsnet_practical_runs/20260627T090323Z_demo_collection/
```

That was intentional: `/tmp` is a temporary location for generated output. It avoids cluttering the repository with run artifacts while we test scripts.

If you want to see the same structure inside the project, run the workflow with:

```bash
python scripts/runnable_workflows/09_practical_collection_workflow.py   --run-label demo_collection   --outdir data/runs
```

Then the run folder will be created under:

```text
data/runs/
```

In real projects, generated run folders can become large. Decide deliberately whether they should be committed to Git, ignored, archived, or stored elsewhere.

## 2. The Run Folder Structure

A practical collection workflow should create one folder per run.

Example:

```text
data/runs/20260703T090000Z_demo_collection/
  config/
    config_snapshot.json
  raw/
    demo_raw_records.json
  processed/
    records.csv
  logs/
    run.log
  reports/
    monitoring_report.json
    version_info.json
    scheduling_examples.md
    manifest.json
```

Each folder has a different purpose. Keeping these purposes separate makes the workflow easier to inspect and debug.

In [ ]:
from pathlib import Path

# This is the location from the earlier test run. The exact timestamp will differ
# on your machine if you run the workflow again.
tmp_parent = Path('/tmp/methodsnet_practical_runs')

if tmp_parent.exists():
    run_folders = sorted(tmp_parent.glob('*'))
    print('Existing demo run folders in /tmp:')
    for folder in run_folders[-5:]:
        print('-', folder)
else:
    print('No /tmp demo run folder found. Run 09_practical_collection_workflow.py to create one.')

## 3. What Each File Means

### `config/config_snapshot.json`

This records the parameters for the run: run label, input file, required columns, minimum row count, and the command that would be scheduled.

Why it matters: if a run produces unexpected output, you can see what settings produced it.

### `raw/demo_raw_records.json`

This is source-shaped evidence. In the no-network demo, it contains synthetic raw records. In a real scraper, this would usually be raw HTML, raw JSON, or a reference to an input file.

Why it matters: raw evidence lets you revisit parsing choices later.

### `processed/records.csv`

This is the analysis-shaped table. It is easier to analyze than raw HTML/JSON, but it is also a transformation.

Why it matters: processed data are useful, but they are lossy. Keep raw data too.

### `logs/run.log`

This records what happened while the script ran: start time, paths, row counts, monitoring status, and completion.

Why it matters: scheduled scripts often run when nobody is watching the terminal.

### `reports/monitoring_report.json`

This records operational checks: row count, required columns, and status-code distribution if available.

Why it matters: monitoring detects likely breakage before bad data quietly enter analysis.

### `reports/version_info.json`

This records Python version, operating system, Git commit, Git status, and package versions.

Why it matters: code and package versions can change results.

### `reports/scheduling_examples.md`

This contains templates for cron, launchd, and GitHub Actions.

Why it matters: scheduling should be documented and cautious.

### `reports/manifest.json`

This is the run-level provenance file. It lists parameters, outputs, checksums, and notes.

Why it matters: the manifest is the map of the run.

## 4. Scheduling Collectors

Scheduling means running a script automatically at a defined time.

Common tools:

- **cron**: common on Linux/macOS;
- **launchd**: macOS-native scheduler;
- **GitHub Actions**: scheduled jobs in a GitHub repository;
- institutional servers or workflow managers for larger projects.

Scheduling is useful when the research question requires repeated collection. But scheduling also increases risk: a broken collector can repeatedly produce bad data or send too many requests.

## 5. Cron Jobs

Cron uses five time fields plus a command:

```text
minute hour day-of-month month day-of-week command
```

Example: run every Monday at 09:00:

```cron
0 9 * * 1 cd /path/to/repo && python scripts/runnable_workflows/03b_methodsnet_course_scraper.py --details 3 --outdir data >> data/cron.log 2>&1
```

Meaning:

- `0`: minute 0;
- `9`: hour 9;
- `*`: every day of the month;
- `*`: every month;
- `1`: Monday;
- `cd /path/to/repo`: move into the project folder;
- `python ...`: run the collector;
- `>> data/cron.log 2>&1`: append normal output and errors to a log file.

Important: cron has a minimal environment. The `python` command in cron may not be the same Python you use in VS Code or Anaconda.

## 6. macOS launchd

On macOS, `launchd` is often preferred over cron.

A LaunchAgent is a `.plist` file usually stored in:

```text
~/Library/LaunchAgents/
```

It can specify:

- the command to run;
- when to run it;
- where to write stdout and stderr logs.

You do not need to memorize launchd XML. The key point is that the scheduled job should still call the same reproducible command-line workflow and write logs/reports.

In [ ]:
launchd_example = '<?xml version="1.0" encoding="UTF-8"?>\n<!DOCTYPE plist PUBLIC "-//Apple//DTD PLIST 1.0//EN"\n "http://www.apple.com/DTDs/PropertyList-1.0.dtd">\n<plist version="1.0">\n<dict>\n  <key>Label</key>\n  <string>org.methodsnet.collection</string>\n  <key>ProgramArguments</key>\n  <array>\n    <string>/bin/zsh</string>\n    <string>-lc</string>\n    <string>cd /path/to/repo && python scripts/runnable_workflows/03b_methodsnet_course_scraper.py --details 3 --outdir data</string>\n  </array>\n  <key>StartCalendarInterval</key>\n  <dict>\n    <key>Weekday</key><integer>1</integer>\n    <key>Hour</key><integer>9</integer>\n    <key>Minute</key><integer>0</integer>\n  </dict>\n  <key>StandardOutPath</key>\n  <string>/path/to/repo/data/launchd_stdout.log</string>\n  <key>StandardErrorPath</key>\n  <string>/path/to/repo/data/launchd_stderr.log</string>\n</dict>\n</plist>'

print(launchd_example[:900])

## 7. GitHub Actions

GitHub Actions can run scheduled jobs in a repository.

This is useful when:

- the project is already on GitHub;
- the workflow can run on a clean machine;
- dependencies are documented;
- secrets are stored as GitHub secrets;
- output storage and retention are planned.

It is not a magic solution. The same access, rate-limit, ethics, and monitoring questions still apply.

In [ ]:
github_actions_example = 'name: Scheduled collection\n\non:\n  schedule:\n    - cron: "0 7 * * 1"\n  workflow_dispatch:\n\njobs:\n  collect:\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: "3.11"\n      - run: pip install -r requirements.txt\n      - run: python scripts/runnable_workflows/03b_methodsnet_course_scraper.py --details 3 --outdir data'

print(github_actions_example)

## 8. Monitoring Breakage

Monitoring asks whether a run looks operationally plausible.

It is not the same as substantive validation. It does not tell us whether the research design is good. It tells us whether the collector may have broken.

Common monitoring checks:

- row count is not zero;
- row count is not unexpectedly low/high;
- required columns exist;
- key fields are not suddenly missing;
- status codes are mostly `2xx`;
- selectors still match expected records;
- raw response shape did not change;
- output files were actually written.

In [ ]:
import pandas as pd

# A tiny demo table, similar to what a collector might produce.
records = pd.DataFrame(
    [
        {"record_id": "demo-001", "title": "First record", "url": "https://example.org/1", "status_code": 200},
        {"record_id": "demo-002", "title": "Second record", "url": "https://example.org/2", "status_code": 200},
    ]
)

required_columns = ["record_id", "title", "url"]
checks = []

checks.append({
    "check": "minimum_row_count",
    "passed": len(records) >= 1,
    "observed": len(records),
    "expected": ">= 1",
})

for column in required_columns:
    checks.append({
        "check": f"required_column:{column}",
        "passed": column in records.columns,
        "observed": ",".join(records.columns),
        "expected": column,
    })

checks.append({
    "check": "status_codes_are_2xx",
    "passed": records["status_code"].astype(str).str.startswith("2").all(),
    "observed": records["status_code"].value_counts().to_dict(),
    "expected": "all 2xx",
})

pd.DataFrame(checks)

## 9. Logging

Logs are for events, warnings, and errors.

A good `run.log` might say:

```text
2026-07-03T09:00:00 INFO Starting collection
2026-07-03T09:00:02 INFO Fetched course list
2026-07-03T09:00:04 INFO Parsed 26 course links
2026-07-03T09:00:04 WARNING Expected at least 30 links, found 26
2026-07-03T09:00:05 INFO Saved processed CSV
```

Logs are especially important for scheduled scripts because there may be no visible terminal output.

In [ ]:
import logging
from pathlib import Path

example_log_path = Path('/tmp/methodsnet_example_run.log')
logger = logging.getLogger('course_demo_logger')
logger.setLevel(logging.INFO)
logger.handlers.clear()

handler = logging.FileHandler(example_log_path, encoding='utf-8')
handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
logger.addHandler(handler)

logger.info('Starting example run')
logger.info('Parsed %s records', len(records))
logger.info('Monitoring checks passed: %s', all(check['passed'] for check in checks))

print(example_log_path)
print(example_log_path.read_text())

## 10. Versioning

Versioning records the computational context of a run.

At minimum, record:

- script path;
- command-line arguments or config file;
- Git commit hash;
- Git status;
- Python version;
- package versions.

The Git status matters because a commit hash alone is not enough if there were uncommitted local changes.

In [ ]:
import platform
import subprocess
import sys


def safe_command(command):
    try:
        result = subprocess.run(command, check=True, capture_output=True, text=True)
        return result.stdout.strip()
    except Exception:
        return None

version_info = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "git_commit": safe_command(["git", "rev-parse", "HEAD"]),
    "git_status_short": safe_command(["git", "status", "--short"]),
}

version_info

## 11. Making Collectors Reproducible

A reproducible collector should answer:

- What command was run?
- With which parameters?
- On which code version?
- In which environment?
- What raw evidence was saved?
- What processed data were produced?
- What checks passed or failed?
- What limitations should future users know?

This is why the practical workflow writes a `manifest.json`. The manifest does not replace the data. It explains the data.

## 12. Practical Workflow Command

The repository includes a wrapper script for this operational layer:

```bash
python scripts/runnable_workflows/09_practical_collection_workflow.py   --run-label demo_collection   --outdir data/runs
```

It creates a run folder with:

```text
config/config_snapshot.json
raw/demo_raw_records.json
processed/records.csv
logs/run.log
reports/monitoring_report.json
reports/version_info.json
reports/scheduling_examples.md
reports/manifest.json
```

To monitor an existing processed CSV, use:

```bash
python scripts/runnable_workflows/09_practical_collection_workflow.py   --run-label methodsnet_course_monitor   --input-csv data/processed/methodsnet_course_links.csv   --required-columns course_url,course_code,title_guess,status_guess   --min-rows 5   --collector-command "python scripts/runnable_workflows/03b_methodsnet_course_scraper.py --details 3 --outdir data"   --outdir data/runs
```

## 13. Exercise

Choose one collector from the course scripts.

Answer:

1. What would be a reasonable run frequency, if any?
2. What would be irresponsible to schedule?
3. Which outputs should go into `raw/`, `processed/`, `logs/`, and `reports/`?
4. Which monitoring checks would catch breakage?
5. Which fields belong in the manifest?
6. Would you commit the run outputs to Git, ignore them, or archive them elsewhere?

## Key Takeaway

Scheduling is not the beginning of automation. It is the end of a careful workflow.

First make the collector explicit, limited, logged, monitored, versioned, and reproducible. Then decide whether scheduling is justified.